<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Feature Detection and Object Tracking</b></h1>
</div>

## Context

Feature-based tracking estimates object motion from repeatable local image structures rather than from a learned temporal model. The object of interest is initialized manually in the first video frame and subsequently localized through local feature correspondences and projective geometry.

The technical challenge is to distinguish descriptor similarity from geometrically valid motion. ORB features provide compact binary descriptors, while RANSAC-based homography estimation is used to reject inconsistent correspondences and determine whether the observed support is sufficient for a valid projective update.

## Problem Statement

Track the initialized object throughout the video using the first frame as a fixed reference, ORB descriptors for appearance matching, Hamming distance for correspondence search, and a RANSAC-estimated homography for geometric localization.

The tracker must:

- validate the input video and initial object region;
- extract reference features strictly inside the initialized object support;
- establish descriptor correspondences in each subsequent frame;
- reject frames that do not provide sufficient geometric support;
- estimate a valid homography only from RANSAC-consistent matches;
- propagate the original four-corner object region through the estimated projective transformation;
- quantify match count, inlier count, inlier ratio, and sequence-level tracking success;
- generate representative tracking and inlier diagnostics;
- validate all numerical outputs and required figures.

Optical flow, learned descriptors, temporal motion models, non-rigid deformation models, and multi-object tracking are outside scope.

## Inputs and Fixed Parameters

| Parameter | Fixed value |
| --- | --- |
| Input video | `data/video1.mp4` |
| Initial row | 24 px |
| Initial column | 46 px |
| Bounding-box height | 170 px |
| Bounding-box width | 160 px |
| Detector / descriptor | ORB |
| Maximum ORB features | 1000 |
| Descriptor length | 32 bytes |
| Matcher | brute force |
| Distance | Hamming |
| Cross-check | enabled |
| Geometric model | $3\times3$ homography |
| Robust estimator | RANSAC |
| RANSAC reprojection threshold | 5.0 px |
| Minimum descriptor matches | 4 |
| Minimum RANSAC inliers | 4 |
| Required outputs | five diagnostic PNG figures |

A frame is considered valid only when the correspondence set supports a finite projective transform and the transformed object polygon is finite.

## 1. Validate the Input Video and Output Paths

1. Verify that `data/video1.mp4` exists and can be opened by OpenCV.
2. Read video metadata and report frame count, frame width, frame height, and frame rate when available.
3. Create `outputs/figures/` if needed.
4. Abort with an explicit diagnostic if the video cannot be decoded.

No tracking step may begin before these checks succeed.

## 2. Define the Initial Bounding Box and Tracking Parameters

Using

$$
r_0=24,\quad c_0=46,\quad h=170,\quad w=160,
$$

construct the ordered four-corner polygon of the initial object region.

Verify that all four corners lie inside the reference-frame dimensions.

Define and report the fixed tracking parameters:

- ORB feature budget = 1000;
- Hamming-distance matcher with cross-check;
- RANSAC reprojection threshold = 5.0 px;
- minimum matches = 4;
- minimum RANSAC inliers = 4.

These values remain unchanged for the full sequence.

## 3. Initialize ORB and the Hamming-Distance Matcher

Instantiate:

1. ORB with a maximum of 1000 detected features;
2. a brute-force descriptor matcher using Hamming distance;
3. mutual cross-checking.

Confirm that the matcher is compatible with binary ORB descriptors.

Report the descriptor dimensionality and explain why Euclidean distance is not used for the binary representation.

## 4. Read and Validate the Reference Frame

Read frame 0 and convert it to grayscale.

Verify:

- the frame is non-empty;
- grayscale dimensions match the color-frame dimensions;
- the initial ROI lies entirely within the frame;
- the recorded total frame count is positive.

Store only the reference data required for subsequent matching; do not retain the entire video in memory.

## 5. Detect Reference ORB Features Inside the Object Region

Create a binary ROI mask equal to one inside the initialized object polygon and zero outside it.

Detect ORB keypoints and descriptors using this mask so that the reference descriptor set is restricted to the object support.

Verify:

- every retained keypoint lies inside the ROI;
- descriptor matrix width is 32 bytes;
- at least four descriptors are available.

Report the number of reference keypoints and descriptors.

## 6. Visualize the Reference Object and ORB Keypoints

Produce `reference_orb_keypoints.png` containing:

- the reference frame;
- the initialized object polygon;
- the detected ORB keypoints, including scale/orientation where supported.

The figure must make it possible to judge whether the selected object region contains enough distinctive local structure for matching.

## 7. Define Frame Matching and RANSAC Homography Estimation

Implement one frame-processing operation that:

1. detects current-frame ORB keypoints/descriptors;
2. matches the fixed reference descriptors to the current descriptors;
3. rejects the frame if fewer than four matches are available;
4. builds paired point coordinates;
5. estimates a homography with RANSAC using a 5.0 px reprojection threshold;
6. counts the RANSAC inliers;
7. rejects the frame if fewer than four inliers remain;
8. returns the homography, matches, inlier mask, and rejection reason.

No fallback bounding box may be fabricated for rejected frames.

## 8. Track the Object Throughout the Video

Process every frame after the reference frame.

For each accepted homography $H_t$, transform the four reference corners:

$$
s\mathbf x_t
=
H_t\mathbf x_0.
$$

Reject any frame whose projected polygon contains non-finite coordinates.

Store compact per-frame records containing:

- frame index;
- transformed polygon;
- raw match count;
- RANSAC inlier count;
- inlier ratio;
- success/rejection status.

Do not store full decoded video frames solely for later plotting.

## 9. Compute Tracking Summary Metrics

For all processed frames, compute:

$$
\mathrm{SuccessRate}
=
\frac{N_{accepted}}{N_{processed}},
$$

and, over accepted frames:

- mean/median descriptor-match count;
- mean/median RANSAC inlier count;
- mean/median inlier ratio

$$
r_t=
\frac{N_{inliers,t}}{N_{matches,t}}.
$$

Also report the number of rejected frames and the dominant rejection reasons.

## 10. Visualize Representative Tracking Frames

Select a small set of successful frames distributed across the sequence and reload only those frames from disk/video.

Overlay the transformed object polygon and frame index.

Save the montage as `representative_tracking_frames.png`.

The selected frames must be temporally distributed rather than chosen only from the beginning of the sequence.

## 11. Visualize RANSAC Inlier Matches

Select one representative successful frame and recompute its full correspondence set.

Display only the matches classified as RANSAC inliers between the reference ROI and the selected current frame.

Save the diagnostic as `ransac_inlier_matches.png`.

Report:

- total matches;
- inlier matches;
- inlier ratio;
- estimated homography.

The figure must support visual inspection of geometric consistency.

## 12. Analyze Matches, Inliers, and Inlier Ratio Across the Sequence

Generate two sequence-level diagnostics:

1. `matches_and_inliers_by_frame.png`  
   Plot raw descriptor matches and RANSAC inliers versus frame index.

2. `inlier_ratio_by_frame.png`  
   Plot

$$
r_t=
\frac{N_{inliers,t}}{N_{matches,t}}
$$

for accepted frames.

Identify frames with abrupt degradation and relate them to correspondence support rather than assuming every successful homography has equal reliability.

## 13. Run Numerical and Output-file Validation Checks

Verify:

1. reference descriptor count is at least four;
2. reference descriptors have the expected binary-descriptor width;
3. processed-frame accounting matches the video length;
4. every accepted homography is finite and $3\times3$;
5. every accepted transformed box contains four finite 2-D corners;
6. $N_{inliers}\le N_{matches}$ for every frame;
7. every inlier ratio lies in $[0,1]$;
8. rejected frames contain no fabricated geometry;
9. all five diagnostic figures exist.

Any failed check must identify the frame or artifact responsible.

## Completion Criterion

The laboratory is complete when the object has been processed across the full video, every accepted localization is supported by a valid RANSAC homography, rejected frames are handled explicitly, sequence-level correspondence diagnostics are reported, and all five required figures and numerical checks are complete.